In [ ]:
import pandas as pd
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))
from config.constants import PROJECT_ROOT
import numpy as np
import pickle
from scipy.stats import spearmanr

%load_ext autoreload
%autoreload 2

### select rep cohorts

In [ ]:
rep_name = "representative_cohorts_K1000_96e897.txt"
rep_cohorts = pd.read_csv(f"../saved_data/rep_selections/{rep_name}",header=None,names=["cohort_name"])

# Load results folds features

In [ ]:
RESULTS_DIR = Path(PROJECT_ROOT) / "saved_data"  / "results"
METRICS = ["auroc", "auprc", "f1", "mcc", "recall", "precision"]

def load_results(cohorts, prefix, agg,variant, models=("random_forest",), folds=(0,),results_dir=RESULTS_DIR):
    rows, missing = [], []
    for cohort in cohorts:
        for model in models:
            for f in folds:
                file = results_dir / cohort / model / prefix / f"fold_{f}" / "results_final.csv"
                if not file.exists():
                    missing.append(cohort)
                    continue
                    
                rows.append(pd.read_csv(file).assign(
                    cohort_name=cohort, model=model, prefix=prefix, fold=f))
    if not rows:
        return pd.DataFrame()
    df = pd.concat(rows, ignore_index=True)
    df[METRICS] = df[METRICS].apply(pd.to_numeric, errors="coerce")
    df = df[["cohort_name"] + METRICS]
    df.attrs["missing"] = missing
    return df


In [ ]:
def load_folds(cohort, fold=0, seed=42, first_adm_only=True):
    suffix = "_firstadm" if first_adm_only else ""
    f = Path(PROJECT_ROOT) / "saved_data" / "folds" / cohort / f"seed_{seed}{suffix}" / f"fold_{fold}.pkl"
    if not f.exists():
        return None#
    with open(f, "rb") as fh:
        train, val, test = pickle.load(fh)
    ids = {"train": train[:, 1], "val": val[:, 1], "test": test[:, 1]}
    
    return ids

def load_cohort(cohort, fold=0, seed=42):
    df = pd.read_csv(Path(PROJECT_ROOT) / "saved_data" / "cohorts" / "DTB" / "new" / f"{cohort}.csv.gz",compression="gzip")
    df = (df.sort_values("admittime", kind="stable").drop_duplicates("subject_id", keep="first"))
    return df[['subject_id', 'hadm_id', 'admittime', 'dischtime','gender', 'age','dod','D1_date', 'D1_W', 'D1_5y','D2_date', 'label']]

def load_features(cohort, days=14):
    df = pd.read_csv(Path(PROJECT_ROOT) / "saved_data" / "features" / f"features_{days}d_firstadm" / cohort / "features.csv.gz",
                       dtype={"subject_id": "int64", "hadm_id": "int64", "minute": "int64",
                              "itemid": "string", "value": "float64"})
    return df

---


----

## Cohort Characteristics

In [ ]:
cohort_cc_14d = pd.read_csv( Path(PROJECT_ROOT) / "saved_data" / "characteristics"/"characteristics_14d_agg24_top100_train+val.csv")
cohort_cc_180d = pd.read_csv( Path(PROJECT_ROOT) / "saved_data" / "characteristics"/"characteristics_180d_agg24_top100_train+val.csv")


# Check results

In [ ]:
RUNS = {
    "14d daily VMD RF":     dict(prefix="first_adms_14d_24h",          agg=24,   variant="VMD"),
    "180d daily VMD RF":    dict(prefix="first_adms_180d_mimic100",    agg=24,   variant="VMD"),
    "14d 1 bin stats RF":   dict(prefix="stats_14d_336h",              agg=336,  variant="stats"),
    "14d 1 bin count RF":   dict(prefix="count_14d_336h",              agg=336,  variant="count"),
    "180d 3 bins stats RF": dict(prefix="stats_180d_1440h",            agg=1440, variant="stats"),
    "180d 6 bins stats RF": dict(prefix="stats_180d_720h",             agg=720, variant="stats"),
    "180d 3 bins count RF": dict(prefix="count_180d_1440h",            agg=1440, variant="count"),
    "14d daily VMD LR":     dict(prefix="first_adms_14d_24h",          agg=24,   variant="VMD",   models=("logistic_regression",)),
    "14d 1 bin stats LR":   dict(prefix="stats_14d_336h",              agg=336,  variant="stats", models=("logistic_regression",)),
    "14d 1 bin count LR":   dict(prefix="count_14d_336h",              agg=336,  variant="count", models=("logistic_regression",)),
    "180d 3 bins stats LR": dict(prefix="stats_180d_1440h",            agg=1440, variant="stats", models=("logistic_regression",)),
    "180d 6 bins stats LR": dict(prefix="stats_180d_720h",             agg=720, variant="stats", models=("logistic_regression",)),
    "14d daily VMD CB":     dict(prefix="first_adms_14d_24h",       agg=24,   variant="VMD",   models=("catboost",)),
    "180d daily VMD CB":    dict(prefix="first_adms_180d_mimic100", agg=24,   variant="VMD",   models=("catboost",)),
    "14d 1 bin stats CB":   dict(prefix="stats_14d_336h",           agg=336,  variant="stats", models=("catboost",)),
    "14d 1 bin count CB":   dict(prefix="count_14d_336h",           agg=336,  variant="count", models=("catboost",)),
    "180d 3 bins stats CB": dict(prefix="stats_180d_1440h",         agg=1440, variant="stats", models=("catboost",)),
    "180d 6 bins stats CB": dict(prefix="stats_180d_720h",         agg=720, variant="stats", models=("catboost",)),
}

hpc_run_dir = Path(PROJECT_ROOT) / "saved_data" / "results"

rows = []
for name, cfg in RUNS.items():
    df = load_results(rep_cohorts["cohort_name"], results_dir=hpc_run_dir, **cfg)
    n_missing = len(df.attrs.get("missing", []))
    rows.append(df.assign(run=name, feat=cfg["variant"],
                          model=cfg.get("models", ("random_forest",))[0]))
    print(f"{name:24s} {len(df):4d} cohorts, {n_missing:4d} missing")

runs_df = pd.concat(rows, ignore_index=True)



In [ ]:
RUNS_DL = {
    #"14d daily VMD LSTM":       dict(prefix="first_adms_14d_24h",    agg=24,  variant="VMD",   models=("lstm",)),
    #"180d daily VMD LSTM":      dict(prefix="first_adms_180d_24h",   agg=24,  variant="VMD",   models=("lstm",)),
    #"180d 18 bins stats LSTM":  dict(prefix="stats_180d_240h_grid",  agg=240, variant="stats", models=("lstm",)),
    "180d 6 bins stats LSTM":   dict(prefix="stats_180d_720h_grid",  agg=720, variant="stats", models=("lstm",)),
    #"14d daily VMD GRU":        dict(prefix="first_adms_14d",        agg=24,  variant="vmd",   models=("gru",)),
    #"180d daily VMD GRU":       dict(prefix="first_adms_180d",       agg=24,  variant="vmd",   models=("gru",)),
    #"180d 18 bins stats GRU":   dict(prefix="stats_180d_240h",       agg=240,  variant="stats",   models=("gru",)),
    #"180d 6 bins stats GRU":    dict(prefix="stats_180d_720h",       agg=720,  variant="stats",   models=("gru",)),
}

rows_dl = []
for name, cfg in RUNS_DL.items():
    df = load_results(rep_cohorts["cohort_name"], results_dir=hpc_run_dir, **cfg)
    rows_dl.append(df.assign(run=name, feat=cfg["variant"], model=cfg["models"][0]))
    print(f"{name:26s} {len(df):4d} cohorts, {len(df.attrs.get('missing', [])):4d} missing")

runs_df = pd.concat([runs_df] + rows_dl, ignore_index=True).drop_duplicates(["cohort_name", "run"], keep="last")


# Plot

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

MODEL_ORDER  = ["RF", "LR", "CB", "LSTM", "GRU"]
MODEL_NAME   = {"RF": "Random Forest", "LR": "Logistic Regression", "CB": "CatBoost",
                "LSTM": "LSTM", "GRU": "GRU"}
MODEL_COLORS = {
    "RF":   "#7FA6C9",
    "LR":   "#82B5A5",
    "CB":   "#9CC9DC",
    "LSTM": "#D58B8B",
    "GRU":  "#D6A080",
}

GROUPS      = ["VMD", "stats", "count"]
GROUP_LABEL = {"VMD": "daily (VMD)", "stats": "stats", "count": "count"}
FEAT_MARKER = {"VMD": "o", "stats": "s", "count": "^"}
FEAT_NAME   = {"VMD": "daily (VMD)", "stats": "stats", "count": "count"}

WIN_ORDER = ["14d daily", "14d 1 bin", "180d daily", "180d 3 bins", "180d 6 bins", "180d 18 bins"]

PLOT_RUNS = None
#PLOT_RUNS = [r for r in runs_df.run.unique() if "daily" in r]
df   = runs_df if PLOT_RUNS is None else runs_df[runs_df.run.isin(PLOT_RUNS)]
runs = dict(tuple(df.groupby("run")))

def feat_of(label):
    for k in GROUPS:
        if k in label:
            return k
    return None

def model_of(label):
    return label.rsplit(" ", 1)[-1]

def win_of(label):
    return next(w for w in WIN_ORDER if label.startswith(w))

def within_key(label):
    return (WIN_ORDER.index(win_of(label)), MODEL_ORDER.index(model_of(label)))

grouped = {g: sorted([k for k in runs if feat_of(k) == g], key=within_key) for g in GROUPS}

GAP    = 1.0
SUBGAP = 0.45
pos, labels, keys, group_span, win_split = [], [], [], {}, []
x = 1.0
for g in GROUPS:
    members = grouped[g]
    if not members:
        continue
    start = x
    prev  = None
    for k in members:
        if prev is not None and prev.startswith("14d") and k.startswith("180d"):
            win_split.append(x - 0.5 + SUBGAP / 2)
            x += SUBGAP
        pos.append(x); keys.append(k)
        labels.append(k.replace(f" {g}", "").replace("VMD ", ""))
        prev = k
        x += 1.0
    group_span[g] = (start, x - 1.0)
    x += GAP

metrics = ["auroc", "auprc", "f1"]

for m in metrics:
    fig, ax = plt.subplots(figsize=(11, 7))

    data = [runs[k][m].dropna().values for k in keys]
    bp = ax.boxplot(data, positions=pos, widths=0.62, patch_artist=True, showcaps=False,
                    medianprops=dict(color="#545252", linewidth=0.6),
                    whiskerprops=dict(color="#52514e", linewidth=1),
                    flierprops=dict(marker=".", markersize=2, alpha=0.25,
                                    markeredgecolor="none", markerfacecolor="#52514e"))
    for patch, k in zip(bp["boxes"], keys):
        patch.set(facecolor=MODEL_COLORS[model_of(k)], edgecolor="#fcfcfb", linewidth=2)

    # median marker encodes the feature type
    for p, k, d in zip(pos, keys, data):
        ax.plot(p, np.median(d), marker=FEAT_MARKER[feat_of(k)], markersize=6,
                markerfacecolor="#2f2d2d", markeredgecolor="#0b0b0b",
                markeredgewidth=0, zorder=5, linestyle="none")

    ax.set_title(m.upper(), color="#0b0b0b", fontsize=22, pad=34)
    ax.set_xticks(pos)
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=15, color="#52514e")
    ax.set_xlim(0.3, x - GAP + 0.7)
    ax.grid(axis="y", color="#e5e4e0", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(axis="y", labelsize=18, colors="#52514e")
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)

    for g, (lo, hi) in group_span.items():
        ax.text((lo + hi) / 2, 1.015, GROUP_LABEL[g], transform=ax.get_xaxis_transform(),
                ha="center", va="bottom", fontsize=17, color="#52514e", fontweight="bold")
        ax.plot([lo - 0.4, hi + 0.4], [1.005, 1.005], transform=ax.get_xaxis_transform(),
                color="#52514e", lw=2.5, clip_on=False)
    for g1, g2 in zip(GROUPS, GROUPS[1:]):
        if g1 in group_span and g2 in group_span:
            ax.axvline((group_span[g1][1] + group_span[g2][0]) / 2,
                       color="#e5e4e0", lw=1.2, zorder=0)
    for xs in win_split:
        ax.axvline(xs, color="#b8b8b8", lw=1.2, ls=":", zorder=0)

    model_handles = [Patch(facecolor=MODEL_COLORS[mm], edgecolor="#fcfcfb", lw=2,
                           label=MODEL_NAME[mm]) for mm in MODEL_ORDER]
    feat_handles = [Line2D([], [], marker=FEAT_MARKER[g], linestyle="none", markersize=6,
                           markerfacecolor="#0b0b0b", markeredgecolor="#0b0b0b",
                           label=FEAT_NAME[g]) for g in GROUPS]

    leg1 = fig.legend(handles=model_handles, loc="lower center", ncol=5, frameon=False,
                      fontsize=14, bbox_to_anchor=(0.5, -0.03), title="model")
    fig.legend(handles=feat_handles, loc="lower center", ncol=3, frameon=False,
               fontsize=14, bbox_to_anchor=(0.5, -0.13), title="features")
    fig.add_artist(leg1)

    fig.tight_layout(rect=[0, 0.16, 1, 1])
    plt.show()

# table view
mean  = pd.DataFrame({k: runs[k][metrics].mean() for k in keys}).T
std   = pd.DataFrame({k: runs[k][metrics].std()  for k in keys}).T
order = mean.sort_values("auroc", ascending=False).index
tbl   = (mean.round(3).astype(str) + " ± " + std.round(3).astype(str)).loc[order]
tbl


In [ ]:
def show(metric="auroc", runs=None, per_cohort=False):
    df = runs_df if runs is None else runs_df[runs_df["run"].isin(runs)]
    if per_cohort:
        return df.pivot_table(index="cohort_name", columns="run", values=metric)
    return (df.groupby("run")[metric]
              .agg(n="count", mean="mean", std="std", median="median")
              .sort_values("mean", ascending=False))
#show("auroc")                                    # all runs, summary

runs_180d_6bins = ["180d 6 bins stats GRU",
               "180d 6 bins stats LSTM",
               "180d 6 bins stats RF",
               "180d 6 bins stats CB",
               "180d 6 bins stats LR"]

runs_14d_daily = ["14d daily VMD RF",
                  "14d daily VMD LR",
                  "14d daily VMD CB",
                  "14d daily VMD LSTM",
                  "14d daily VMD GRU"]


show("auroc", runs_14d_daily)   



# Characteristics

In [ ]:
#cohens_d_adj_z — How different the average lab values are between pos and neg patients, compared with what could happen by chance.
#h_diff — How much less spread out the lab values are in pos patients compared with neg patients.

CHARACTERISTICS = [
    "log10_n", "target_rate",                              # size and prevalence
    "cohens_d_adj_z", "h_diff",                      # separability ()
    "n_events_mean",                               # data volume
    "log2_events_ratio",                             # are positives measured more than negatives
    "itemid_missing_mean",                           # panel coverage, binning-independent
    "adm_missing_std",                               # heterogeneity across admissions
    "itembin_missing_mean",                          # temporal density (computed at agg_int=24h)
    "itembin_missing_pos_minus_neg",                 # informative missingness
    "age", "female_rate", "death_rate", "log10_RR", "W",        # population
]

CHAR_LABELS = {
    "log10_n":                       "Log Cohort size",
    "target_rate":                   "Target rate",
    "cohens_d_adj_z":                "Cohen's d (z vs null)",
    "h_diff":                        "Entropy diff (neg - pos)",
    "n_events_mean":                 "Mean events/adm",
    "log2_events_ratio":             "log2(events pos/neg)",
    "itemid_missing_mean":           "ItemId missingness per adm",
    "adm_missing_std":               "Adm missingness SD per ItemID",
    "itembin_missing_mean":          "Item-bin matrix missingness",
    "itembin_missing_pos_minus_neg": "Item-bin missingness (pos - neg)",
    "age":                           "Age",
    "female_rate":                   "Female rate",
    "death_rate":                    "Death rate",
    "RR":                            "Log Relative risk (D2 | D1)",
    "W":                             "D1-D2 gap (days)",
}


# Across all cohort

In [ ]:
METRIC = "auroc"

# --- characteristics table per window, with the derived columns ---
def prep_characteristics(cc):
    c = cc.set_index("cohort_name") if "cohort_name" in cc.columns else cc.copy()
    c["log2_events_ratio"] = np.log2(c.n_events_mean_pos / c.n_events_mean_neg)
    c["itembin_missing_pos_minus_neg"] = c.itembin_missing_mean_pos - c.itembin_missing_mean_neg
    c["log10_n"]  = np.log10(c.n)
    c["log10_RR"] = np.log10(c.RR)
    return c

CHAR_TABLES = {"14d":  prep_characteristics(cohort_cc_14d),
               "180d": prep_characteristics(cohort_cc_180d)}


def characteristic_correlations(run, metric=METRIC, n_bootstrap=1000, seed=0):
    """Spearman rho + D1-cluster-bootstrap CI for every characteristic, one run."""
    window = "180d" if run.startswith("180d") else "14d"
    chars  = CHAR_TABLES[window]

    scores = runs_df.loc[runs_df.run == run].set_index("cohort_name")[metric]
    data   = chars[CHARACTERISTICS].join(scores.rename(metric), how="inner").dropna()

    char_and_metric = np.column_stack([data[CHARACTERISTICS].to_numpy(float),
                                       data[metric].to_numpy(float)])
    rho_observed = spearmanr(char_and_metric)[0][-1, :-1]

    d1_of_cohort = data.index.str.split("-").str[0].to_numpy()
    d1_groups    = np.unique(d1_of_cohort)
    rows_by_d1   = {d1: np.flatnonzero(d1_of_cohort == d1) for d1 in d1_groups}

    rng      = np.random.default_rng(seed)
    rho_boot = np.empty((n_bootstrap, len(CHARACTERISTICS)))
    for i_boot in range(n_bootstrap):
        resampled_d1   = rng.choice(d1_groups, d1_groups.size, replace=True)
        resampled_rows = np.concatenate([rows_by_d1[d1] for d1 in resampled_d1])
        rho_boot[i_boot] = spearmanr(char_and_metric[resampled_rows])[0][-1, :-1]

    ci_low, ci_high = np.nanpercentile(rho_boot, [2.5, 97.5], axis=0)

    return pd.DataFrame({"rho": rho_observed, "ci_low": ci_low, "ci_high": ci_high,
                         "robust": (ci_low > 0) | (ci_high < 0)},
                        index=CHARACTERISTICS).assign(
                            run=run, window=window,
                            n_cohorts=len(data), n_clusters=len(d1_groups))


# --- all runs ---
char_corr = pd.concat(
    [characteristic_correlations(r).rename_axis("characteristic").reset_index()
     for r in sorted(runs_df.run.unique())], ignore_index=True)

# rho per run
rho_matrix = char_corr.pivot(index="characteristic", columns="run", values="rho").round(2)


# how consistent each characteristic is across runs
consistency = (char_corr.groupby("characteristic")
    .apply(lambda g: pd.Series({
        "n_robust":  int(g.robust.sum()),
        "n_runs":    len(g),
        "same_sign": bool(g.loc[g.robust, "rho"].gt(0).nunique() <= 1),
        "rho_mean":  g.rho.mean(),
        "rho_min":   g.rho.min(),
        "rho_max":   g.rho.max(),
    }), include_groups=False)
    .sort_values(["n_robust", "rho_mean"], key=abs, ascending=False).round(3))



In [ ]:
CMAP = "coolwarm"

def run_sort_key(r):
    return (0 if r.startswith("14d") else 1,
            1 if model_of(r) in {"LSTM", "GRU"} else 0, 
            GROUPS.index(feat_of(r)),
            MODEL_ORDER.index(model_of(r)),
            WIN_ORDER.index(win_of(r)))

HEATMAP_RUNS = None
#HEATMAP_RUNS = [r for r in rho_matrix.columns if "daily" in r]

cols      = rho_matrix.columns if HEATMAP_RUNS is None else [c for c in rho_matrix.columns if c in HEATMAP_RUNS]
RUN_ORDER = sorted(cols, key=run_sort_key)

CHAR_ORDER = consistency.index

mat = rho_matrix.loc[CHAR_ORDER, RUN_ORDER]
rob = char_corr.pivot(index="characteristic", columns="run", values="robust").loc[CHAR_ORDER, RUN_ORDER]

split = next((j for j, c in enumerate(mat.columns) if c.startswith("180d")), len(mat.columns))


blocks = [(mat.iloc[:, :split], rob.iloc[:, :split]), (mat.iloc[:, split:], rob.iloc[:, split:])]
vmax   = np.nanmax(np.abs(mat.to_numpy()))

fig, axes = plt.subplots(1, 2, figsize=(1.25*mat.shape[1] + 10, 0.95*len(mat) + 4),
                         sharey=True,
                         gridspec_kw={"width_ratios": [split, mat.shape[1] - split],
                                      "wspace": 0.01})
fig.patch.set_facecolor("white")

for ax, (sub, subrob) in zip(axes, blocks):
    im = ax.imshow(sub, cmap=CMAP, vmin=-vmax, vmax=vmax, aspect="auto")

    ax.set_xticks(np.arange(sub.shape[1] + 1) - 0.5, minor=True)
    ax.set_yticks(np.arange(sub.shape[0] + 1) - 0.5, minor=True)
    ax.grid(which="minor", color="white", linewidth=3)
    ax.tick_params(which="minor", length=0)

    for i in range(sub.shape[0]):
        for j in range(sub.shape[1]):
            v = sub.iat[i, j]
            ax.text(j, i, f"{v:+.2f}", ha="center", va="center", fontsize=28,
                    color=("white" if abs(v) > 0.72*vmax else "#1a1a1a") if subrob.iat[i, j] else "#b4b4b4")

    ax.set_xticks(range(sub.shape[1]))
    ax.set_xticklabels(sub.columns, rotation=45, ha="right", fontsize=30)
    for lbl, c in zip(ax.get_xticklabels(), sub.columns):
        lbl.set_color("#c0392b" if model_of(c) in {"LSTM", "GRU"} else "#3a3a3a")

    ax.tick_params(axis="both", length=0, pad=8)
    for spine in ax.spines.values():
        spine.set_visible(False)
        
    

axes[0].set_yticks(range(len(mat)))
axes[0].set_yticklabels([CHAR_LABELS.get(c, c) for c in mat.index], fontsize=30, color="#1a1a1a")

fig.suptitle(f"Cohort characteristics vs {METRIC.upper()}\n"
             "Spearman rho per run  ·  grey = 95% CI includes zero",
             fontsize=36, y=1.02, color="#1a1a1a")

cb = fig.colorbar(im, ax=axes, fraction=0.018, pad=0.012)
cb.set_label("Spearman rho", fontsize=36, color="#3a3a3a")
cb.ax.tick_params(labelsize=36, length=0, colors="#3a3a3a")
cb.outline.set_visible(False)



In [ ]:
RUN    = "14d daily VMD RF"
METRIC = "auroc"

from statsmodels.nonparametric.smoothers_lowess import lowess

corr   = characteristic_correlations(RUN, METRIC)
chars  = CHAR_TABLES[corr.window.iloc[0]]
scores = runs_df.loc[runs_df.run == RUN].set_index("cohort_name")[METRIC]
data   = chars[CHARACTERISTICS].join(scores.rename(METRIC), how="inner").dropna()

print(f"{corr.n_clusters.iloc[0]} D1 clusters | {corr.n_cohorts.iloc[0]} cohorts | "
      f"{corr.robust.sum()} of {len(CHARACTERISTICS)} characteristics robust")

metric_values = data[METRIC]
ncol = 5
nrow = int(np.ceil(len(CHARACTERISTICS) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.0*ncol, 3.6*nrow), sharey=True, squeeze=False)

for ax, char in zip(axes.ravel(), CHARACTERISTICS):
    char_values = data[char]
    row         = corr.loc[char]

    ax.scatter(char_values, metric_values, s=6,
               color="#2a6bcd" if row.robust else "#b09797", alpha=.25, linewidths=0)

    trend = lowess(metric_values, char_values, frac=0.6, return_sorted=True)
    ax.plot(trend[:, 0], trend[:, 1], color="#a27d76" if row.robust else "#d8d8d8",
            lw=1.8 if row.robust else 1.0, ls="-" if row.robust else "--")

    ax.set_title(f"{CHAR_LABELS.get(char, char)}\n"
                 f"rho={row.rho:+.2f} [{row.ci_low:+.2f},{row.ci_high:+.2f}]", fontsize=16)

    x_low, x_high = char_values.quantile([0.01, 0.99])
    pad = 0.05 * (x_high - x_low)
    ax.set_xlim(x_low - pad, x_high + pad)
    ax.tick_params(labelsize=14)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)

for ax in axes.ravel()[len(CHARACTERISTICS):]:
    ax.axis("off")
for ax in axes[:, 0]:
    ax.set_ylabel(METRIC.upper(), fontsize=15)

fig.suptitle(f"{RUN} — {METRIC.upper()} — {corr.n_cohorts.iloc[0]} cohorts, "
             f"95% CI over {corr.n_clusters.iloc[0]} D1 clusters", y=1.01, fontsize=24)
fig.tight_layout()


# Check feature importances

In [ ]:
import re, pandas as pd
from pathlib import Path

ROOT    = Path("/home/atuin/b310dc/b310dc10/FLabBench-pipeline/saved_data/results")
MODEL   = "random_forest"
CONFIGS = ["first_adms_14d_24h", "first_adms_180d_mimic100",
           "count_14d_336h", "count_180d_1440h",
           "stats_14d_336h", "stats_180d_720h", "stats_180d_1440h"]
pat = re.compile(r"(?P<item>\d+)_(?P<type>[A-Za-z]+)_[Bb]in(?P<bin>\d+)")

parts = []
for cfg in CONFIGS:
    for f in sorted(ROOT.glob(f"*/{MODEL}/{cfg}/fold_0/feature_importances.csv")):
        d = pd.read_csv(f)
        if d.empty:
            continue
        m = d.feature.str.extract(pat)
        d["config"] = cfg
        d["cohort"] = f.parts[-5]
        d["item"]   = m["item"]
        d["type"]   = m["type"]
        d["bin"]    = pd.to_numeric(m["bin"])
        d["imp"]    = d.importance #/ d.importance.sum()   # per-cohort normalized RF alreaddy sum to 1
        parts.append(d)

fi = pd.concat(parts, ignore_index=True)
print(fi.shape)


In [ ]:
#sum of importances for each type(V, M,D, mean, max,...) for each cohort, per config
share = (fi.groupby(["config", "cohort", "type"])["imp"].sum().unstack("type").fillna(0))
share = share.div(share.sum(axis=1), axis=0)    # divide each row by its total so each (config, cohort) row sums to 1 
agg   = share.groupby("config").mean().loc[CONFIGS]   # mean across cohorts


In [ ]:
colors = ["#e79a97", "#93b8d4", "#9ecb9b", "#bfa8d9", "#f0bb7e",
          "#e8d98a", "#cbb394", "#efa7c6", "#a9b0b8"]

ax = agg.plot.bar(stacked=True, figsize=(12, 5), color=colors,edgecolor="white", linewidth=0.8)
ax.legend(title="type", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.set_ylabel("mean share of importance", fontsize=18)
ax.set_xlabel("config", fontsize=20)
ax.tick_params(labelsize=16,rotation = 45)
ax.legend(title="type", fontsize=14, title_fontsize=16,bbox_to_anchor=(1.02, 1), loc="upper left")


# D1 group

In [ ]:
D1 = "F10"


members  = [c for c in rep_cohorts["cohort_name"] if c.split("-")[0] == D1]
df_grp = runs_df[runs_df.cohort_name.isin(members)]
print(f"D1 = {D1}: {len(members)} cohorts")



In [ ]:
all_characteristics = CHAR_TABLES["14d"]
group_characteristics = all_characteristics.loc[members, CHARACTERISTICS]
#how many standard deviations from the average of all 1002 cohorts
group_z = ((group_characteristics - all_characteristics[CHARACTERISTICS].mean())/ all_characteristics[CHARACTERISTICS].std())

characteristic_spread = pd.DataFrame({
    "within_group_sd": group_z.std(),      # 1.0 = group spans the full dataset spread
    "mean_z":          group_z.mean(),     # where the group sits vs all cohorts
    "min":             group_characteristics.min(),
    "max":             group_characteristics.max(),
}).sort_values("within_group_sd", ascending=False)

characteristic_spread.round(3)

In [ ]:
cohort_dfs = {c: load_cohort(c) for c in members}
adms = {c: set(d.hadm_id) for c, d in cohort_dfs.items()}

names = sorted(members, key=lambda c: -len(adms[c]))
O = np.array([[len(adms[a] & adms[b]) / min(len(adms[a]), len(adms[b]))
               for b in names] for a in names])

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(O, cmap="magma_r", vmin=O[O < 1].min(), vmax=1)
ax.set_xticks(range(len(names)), names, rotation=90, fontsize=8)
ax.set_yticks(range(len(names)), names, fontsize=8)
ax.set_title(f"Admission overlap, D1 = {D1}  (range {O[O<1].min():.2f}–1.00)")
fig.colorbar(im, label="overlap")

thr = (O[O < 1].min() + 1) / 2
for i in range(len(names)):
    for j in range(len(names)):
        ax.text(j, i, f"{O[i,j]*100:.0f}", ha="center", va="center", fontsize=6,
                color="black" if O[i,j] < thr else "white")


iu = np.triu_indices(len(names), 1)
print(f"{len(names)} cohorts, pairwise admission overlap: "
      f"median {np.median(O[iu]):.3f}, min {O[iu].min():.3f}, max {O[iu].max():.3f}")


# Compare 180d and 14d features

In [ ]:
common = CHAR_TABLES["14d"].index.intersection(CHAR_TABLES["180d"].index)
a = CHAR_TABLES["14d"].loc[common, CHARACTERISTICS]
b = CHAR_TABLES["180d"].loc[common, CHARACTERISTICS]

window_compare = pd.DataFrame({
    "median_14d":   a.median(),
    "median_180d":  b.median(),
    "median_delta": (b - a).median(),
    "rank_corr":    [spearmanr(a[c], b[c], nan_policy="omit")[0] for c in CHARACTERISTICS],
    "frac_180d_higher": (b > a).mean(),
}).sort_values("rank_corr").round(3)

print(f"{len(common)} cohorts in both tables")
window_compare


In [ ]:
changed = window_compare.index[window_compare.median_delta != 0]

ncol = 4
nrow = int(np.ceil(len(changed) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.6*ncol, 4*nrow), squeeze=False)

for ax, char in zip(axes.ravel(), changed):
    delta = (b[char] - a[char]).dropna()
    lo, hi = delta.quantile([0.01, 0.99])

    ax.hist(delta, bins=40, range=(lo, hi), color="#2a6bcd", alpha=.7)
    ax.axvline(0, color="#52514e", lw=1.2, ls="--")
    ax.axvline(delta.median(), color="#eb6834", lw=1.8)

    row = window_compare.loc[char]
    ax.set_title(f"{CHAR_LABELS.get(char, char)}\n"
                 f"med Δ {row.median_delta:+.3f}  ·  ↑ in {row.frac_180d_higher:.0%} cohorts",
                 fontsize=16)
    ax.set_xlabel("180d − 14d", fontsize=16)
    ax.tick_params(labelsize=14)
    for spine in ("top", "right", "left"):
        ax.spines[spine].set_visible(False)
    ax.set_yticks([])

for ax in axes.ravel()[len(changed):]:
    ax.axis("off")

fig.suptitle(f"Change per cohort, 14d → 180d — {len(common)} cohorts", y=1.01, fontsize=20)
fig.tight_layout()


In [ ]:
import pickle, itertools, numpy as np, pandas as pd

R = Path(PROJECT_ROOT) / "saved_data"
D1_PICK = "F10"

ch  = pd.read_csv(R/"characteristics"/"characteristics_14d_agg24.csv")
ch  = ch[ch.cohort_name.isin(rep_cohorts["cohort_name"])]
grp = ch[ch.D1 == D1_PICK].copy()
members = grp.cohort_name.tolist()

subj = pickle.load(open(R/"subject_sets.pkl", "rb"))

def adm_set(c):
    df = pd.read_csv(R/"cohorts"/"DTB"/"new"/f"{c}.csv.gz",
                     usecols=["subject_id","hadm_id","admittime"])
    df = df.sort_values("admittime", kind="stable").drop_duplicates("subject_id", keep="first")
    return set(df.hadm_id)

adms = {c: adm_set(c) for c in members}
jac = lambda A,B: len(A & B)/len(A | B) if (A or B) else np.nan
ovl = lambda A,B: len(A & B)/min(len(A),len(B)) if (A and B) else np.nan

pairs = pd.DataFrame([
    dict(a=a, b=b,
         subj_jaccard=jac(subj[a],subj[b]), subj_overlap=ovl(subj[a],subj[b]),
         adm_jaccard=jac(adms[a],adms[b]),  adm_overlap=ovl(adms[a],adms[b]))
    for a,b in itertools.combinations(members, 2)])

us, is_ = set().union(*[subj[c] for c in members]), set.intersection(*[subj[c] for c in members])
ua, ia  = set().union(*adms.values()),              set.intersection(*adms.values())

print(f"D1 = {D1_PICK}: {len(members)} cohorts | D2 = {sorted(grp.D2)}")
print(f"  subjects   union {len(us)}  shared by all {len(is_)} ({len(is_)/len(us):.1%})")
print(f"  admissions union {len(ua)}  shared by all {len(ia)} ({len(ia)/len(ua):.1%})")
print(f"  pairwise subject  overlap  median {pairs.subj_overlap.median():.3f}  min {pairs.subj_overlap.min():.3f}")
print(f"  pairwise admission overlap median {pairs.adm_overlap.median():.3f}  min {pairs.adm_overlap.min():.3f}")

grp[["cohort_name","D2","n","n_positive","target_rate","cohens_d_adj",
     "h_diff","itembin_missing_mean","age","death_rate"]].round(3).sort_values("cohort_name")


In [ ]:
long = pd.concat([v.assign(run=k) for k, v in runs.items()])
sub  = long[long.cohort_name.isin(members)]

piv = sub.pivot_table(index="cohort_name", columns="run", values="auroc")
piv = piv[[k for k in keys if k in piv.columns]]      # `keys` = grouped run order from your plot cell
print(f"{piv.shape[0]} cohorts x {piv.shape[1]} runs, missing cells: {piv.isna().sum().sum()}")
piv.round(3)


In [ ]:
runs_grp = {k: v[v.cohort_name.isin(members)] for k, v in runs.items()}
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np

runs_grp = {k: v[v.cohort_name.isin(members)] for k, v in runs.items()}

FEAT_COLORS = {"VMD": "#2a78d6", "stats": "#1baf7a", "count": "#eb6834"}
GROUPS = ["VMD", "stats", "count"]
GROUP_LABEL = {"VMD": "daily (VMD)", "stats": "stats", "count": "count"}

MODEL_ORDER = ["RF", "LR", "CB"]
MODEL_NAME  = {"RF": "Random Forest", "LR": "Logistic Regression", "CB": "CatBoost"}
HATCH       = {"RF": None, "LR": "//", "CB": ".."}
WIN_ORDER   = ["14d daily", "14d 1 bin", "180d 6 bins", "180d 3 bins"]

def feat_of(label):
    for k in GROUPS:
        if k in label:
            return k
    return None

def model_of(label):
    return label.rsplit(" ", 1)[-1]

def win_of(label):
    return next(w for w in WIN_ORDER if label.startswith(w))

def within_key(label):
    return (WIN_ORDER.index(win_of(label)), MODEL_ORDER.index(model_of(label)))

grouped = {g: sorted([k for k in runs_grp if feat_of(k) == g], key=within_key) for g in GROUPS}

GAP = 1.0
pos, labels, keys_g, group_span = [], [], [], {}
x = 1.0
for g in GROUPS:
    members_g = grouped[g]
    if not members_g:
        continue
    start = x
    for k in members_g:
        pos.append(x); keys_g.append(k)
        labels.append(k.replace(f" {g}", "").replace("VMD ", ""))
        x += 1.0
    group_span[g] = (start, x - 1.0)
    x += GAP

metrics = ["auroc", "auprc", "f1"]
fig, axes = plt.subplots(1, len(metrics), figsize=(8.5 * len(metrics), 6.5))

for ax, m in zip(axes, metrics):
    data = [runs_grp[k][m].dropna().values for k in keys_g]
    bp = ax.boxplot(data, positions=pos, widths=0.62, patch_artist=True,
                    showcaps=False, showfliers=False,
                    medianprops=dict(color="#0b0b0b", linewidth=1.6),
                    whiskerprops=dict(color="#52514e", linewidth=1))
    for patch, k in zip(bp["boxes"], keys_g):
        patch.set(facecolor=FEAT_COLORS[feat_of(k)], edgecolor="#fcfcfb", linewidth=2,
                  hatch=HATCH[model_of(k)])

    # overlay the individual cohorts, jittered
    rng = np.random.default_rng(0)
    for p, vals in zip(pos, data):
        ax.scatter(p + rng.uniform(-0.13, 0.13, len(vals)), vals,
                   s=18, color="#0b0b0b", alpha=0.45, zorder=3, linewidths=0)

    ax.set_title(m, color="#0b0b0b", fontsize=20, pad=34)
    ax.set_xticks(pos)
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=15, color="#52514e")
    ax.set_xlim(0.3, x - GAP + 0.7)
    ax.grid(axis="y", color="#e5e4e0", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(axis="y", labelsize=18, colors="#52514e")
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)

    for g, (a, b) in group_span.items():
        ax.text((a + b) / 2, 1.015, GROUP_LABEL[g], transform=ax.get_xaxis_transform(),
                ha="center", va="bottom", fontsize=17, color=FEAT_COLORS[g], fontweight="bold")
        ax.plot([a - 0.4, b + 0.4], [1.005, 1.005], transform=ax.get_xaxis_transform(),
                color=FEAT_COLORS[g], lw=2.5, clip_on=False)
    for g1, g2 in zip(GROUPS, GROUPS[1:]):
        if g1 in group_span and g2 in group_span:
            ax.axvline((group_span[g1][1] + group_span[g2][0]) / 2,
                       color="#e5e4e0", lw=1.2, zorder=0)

handles = [Patch(facecolor="#9a9a9a", edgecolor="#fcfcfb", lw=2, hatch=HATCH[mm],
                 label=MODEL_NAME[mm]) for mm in MODEL_ORDER]
fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False, fontsize=15,
           bbox_to_anchor=(0.5, -0.02))
fig.suptitle(f"D1 = {D1_PICK}  ({len(members)} cohorts)", fontsize=22, y=1.02)
fig.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()


In [ ]:
sina_cohorts = ["C25","C64","C91","E05","E78","F10","I12","I73","J45","K44","M25","M48","O26"]
multi[multi.D1.isin(sina_cohorts)]

# Research questions

Cohorts share patients (`D1-D2` pairs drawn from the same `D1` population), so every test below is clustered by `D1`.
Some characteristics are mechanically linked (`target_rate` -> `auprc`, and `n` / `n_positive` / `counts_cohort` / `W` all track cohort size),
so correlations are reported raw **and** partialled on `n` + `target_rate`.

## Setup: characteristics, results, cluster bootstrap

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import spearmanr

CC    = Path(PROJECT_ROOT) / "saved_data" / "characteristics"
cc14  = pd.read_csv(CC / "characteristics_14d_agg24.csv")
cc180 = pd.read_csv(CC / "characteristics_180d_agg24_top100.csv")
for d in (cc14, cc180):
    d["log2_ratio"] = np.log2(d.n_events_mean_pos / d.n_events_mean_neg)

# results: one row per cohort x run
res = pd.concat([v.assign(run=k) for k, v in runs.items()], ignore_index=True)
res = res.drop_duplicates(["cohort_name", "run"], keep="last")
res["D1"] = res.cohort_name.str.split("-").str[0]
AUROC = res.pivot(index="cohort_name", columns="run", values="auroc")

def cboot(data, stat, B=1000, seed=0):
    """95% CI resampling D1 clusters (cohorts sharing a primary disease share patients)."""
    rng = np.random.default_rng(seed)
    g = data.groupby("D1").indices
    keys = list(g)
    out = []
    for _ in range(B):
        idx = np.concatenate([g[k] for k in rng.choice(keys, len(keys), replace=True)])
        try:
            out.append(stat(data.iloc[idx]))
        except Exception:
            out.append(np.nan)
    return np.nanpercentile(out, [2.5, 50, 97.5])

def pspear(d, x, y, controls):
    """Spearman of x,y with `controls` partialled out (rank residuals)."""
    controls = [c for c in controls if c not in (x, y)]
    if not controls:
        return spearmanr(d[x], d[y], nan_policy="omit").statistic
    r = d[[x, y] + controls].replace([np.inf, -np.inf], np.nan).dropna().rank()
    if len(r) < 10:
        return np.nan
    A = np.c_[np.ones(len(r)), r[controls].values]
    rx = r[x] - A @ np.linalg.lstsq(A, r[x].values, rcond=None)[0]
    ry = r[y] - A @ np.linalg.lstsq(A, r[y].values, rcond=None)[0]
    return np.corrcoef(rx, ry)[0, 1]


def icc(d, col, group="D1"):
    """Fraction of variance in `col` that lies between groups."""
    d = d.dropna(subset=[col])
    g = d.groupby(group)[col]
    between = ((g.mean() - d[col].mean()) ** 2 * g.size()).sum()
    within = ((d[col] - g.transform("mean")) ** 2).sum()
    return between / (between + within)

print(cc14.shape, cc180.shape, AUROC.shape, cc14.D1.nunique(), "D1 groups")

In [ ]:
AUROC

## Q1 - Across cohorts, do positives and negatives have different event counts?

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for row, (name, d) in enumerate([("14d", cc14), ("180d", cc180)]):
    dd = d.dropna(subset=["log2_ratio"])
    ax = axes[row, 0]
    ax.scatter(dd.n_events_mean_neg, dd.n_events_mean_pos, s=8, alpha=.3, lw=0)
    lim = [dd[["n_events_mean_neg", "n_events_mean_pos"]].min().min(),
           dd[["n_events_mean_neg", "n_events_mean_pos"]].max().max()]
    ax.plot(lim, lim, "k--", lw=1)
    ax.set(xscale="log", yscale="log", xlabel="events/adm negatives",
           ylabel="events/adm positives", title=f"{name}: pos vs neg")

    ax = axes[row, 1]
    ax.hist(dd.log2_ratio, bins=60)
    ax.axvline(0, color="k", ls="--", lw=1)
    ax.axvline(dd.log2_ratio.median(), color="r", lw=1)
    ax.set(xlabel="log2(pos/neg events per adm)", ylabel="cohorts", title=f"{name}: ratio")

    lo, med, hi = cboot(dd, lambda s: s.log2_ratio.median())
    flo, fmed, fhi = cboot(dd, lambda s: (s.log2_ratio > 0).mean())
    print(f"{name}: median log2 ratio {med:+.3f} [{lo:+.3f},{hi:+.3f}] (x{2**med:.3f}) | "
          f"frac pos>neg {fmed:.3f} [{flo:.3f},{fhi:.3f}]")
fig.tight_layout()
plt.show()

positives are systematically measured more than negatives.
Observation is informative. The label correlates with how much data exists per patient, independent of what the values are. This is missing-not-at-random by construction, and it's why a pure count model already reaches 0.575–0.588 AUROC.

## Q2 - Where count is predictive (AUROC > 0.60), are events higher in positives or negatives?

In [ ]:
count_runs = {"14d 336h count": cc14, "180d 1440h count": cc180}

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (run, cc) in zip(axes, count_runs.items()):
    d = cc.merge(res[res.run == run][["cohort_name", "auroc", "auprc"]], on="cohort_name")

    d = d.dropna(subset=["log2_ratio", "auroc"])
    d["predictive"] = d.auroc > 0.70

    for flag, lab in [(True, "auroc>0.60"), (False, "auroc<=0.60")]:
        ax.hist(d[d.predictive == flag].log2_ratio, bins=40, alpha=.55, density=True, label=lab)
    ax.axvline(0, color="k", ls="--", lw=1)
    ax.set(xlabel="log2(pos/neg events)", ylabel="density", title=run)
    ax.legend(frameon=False)

    hi_, lo_ = d[d.predictive], d[~d.predictive]
    dm = cboot(d, lambda s: s[s.predictive].log2_ratio.median() - s[~s.predictive].log2_ratio.median())
    rho = cboot(d, lambda s: spearmanr(s.log2_ratio, s.auroc).statistic)
    print(f"\n{run}: {d.predictive.sum()}/{len(d)} predictive")
    print(f"  median log2 ratio  predictive {hi_.log2_ratio.median():+.3f} | rest {lo_.log2_ratio.median():+.3f}"
          f" | diff {dm[1]:+.3f} [{dm[0]:+.3f},{dm[2]:+.3f}]")
    print(f"  frac pos>neg       predictive {(hi_.log2_ratio>0).mean():.3f} | rest {(lo_.log2_ratio>0).mean():.3f}")
    print(f"  spearman(log2_ratio, auroc) {rho[1]:+.3f} [{rho[0]:+.3f},{rho[2]:+.3f}]")
fig.tight_layout()
plt.show()

auroc > 0.60
14d 336h count: 353/1002 predictive
log2_ratio = log2(n_events_mean_pos / n_events_mean_neg) Then take the median across the 353 predictive cohorts (+0.320) and separately across the 649 others (+0.205). 

## Q3 - For which cohorts does 180d training pay off?

In [ ]:
PAIRS = {"stats 180d-14d":     ("180d 1440h stats", "14d 336h stats"),
         "count 180d-14d":     ("180d 1440h count", "14d 336h count"),
         "180d stats-14d VMD": ("180d 1440h stats", "14d agg24 VMD")}

DRIVERS = ["n", "n_positive", "target_rate", "cohens_d_adj_z", "h_diff",
           "n_events_mean", "log2_ratio", "itembin_missing_mean", "W", "age", "death_rate"]

fig, axes = plt.subplots(1, len(PAIRS), figsize=(4.2 * len(PAIRS), 4))
delta_tbl = {}
for ax, (lab, (b, a)) in zip(axes, PAIRS.items()):
    d = AUROC[[a, b]].dropna().assign(delta=lambda x: x[b] - x[a])
    d["D1"] = d.index.str.split("-").str[0]
    d = d.merge(cc180.set_index("cohort_name")[DRIVERS], left_index=True, right_index=True)
    delta_tbl[lab] = d

    ax.hist(d.delta, bins=40)
    ax.axvline(0, color="k", ls="--", lw=1)
    ax.set(xlabel=f"AUROC {b} - {a}", ylabel="cohorts", title=lab)

    w = cboot(d, lambda s: (s.delta > 0).mean())
    m = cboot(d, lambda s: s.delta.mean())
    print(f"\n{lab}: n={len(d)}  win rate {w[1]:.3f} [{w[0]:.3f},{w[2]:.3f}]  "
          f"mean delta {m[1]:+.4f} [{m[0]:+.4f},{m[2]:+.4f}]")

    rows = []
    for f in DRIVERS:
        ci = cboot(d, lambda s, f=f: spearmanr(s[f], s.delta, nan_policy="omit").statistic, B=400)
        rows.append({"feature": f, "rho": ci[1], "lo": ci[0], "hi": ci[2],
                     "rho_partial": pspear(d, f, "delta", ["n", "target_rate"])})
    t = pd.DataFrame(rows).set_index("feature").round(3)
    t["robust"] = t.lo * t.hi > 0
    print(t.reindex(t.rho.abs().sort_values(ascending=False).index).to_string())
fig.tight_layout()
plt.show()

Longer history gives a small, consistent gain concentrated in sparsely measured cohorts", not "180d is the better setup". Longer history the pos/neg ration gets hifger and helps in this regard. 

In [ ]:

lab, drv = "stats 180d-14d", "cohens_d_adj_z"
d = delta_tbl[lab]
q = pd.qcut(d[drv], 10, duplicates="drop")
g = d.groupby(q, observed=True).delta.agg(["mean", "count"])

se = d.groupby(q, observed=True).delta.agg(lambda s: s.std() / np.sqrt(len(s)))
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(len(g)), g["mean"], yerr=1.96 * se.values, capsize=3)
ax.axhline(0, color="k", lw=1)
ax.set_xticks(range(len(g)), [f"{i.mid:.2g}" for i in g.index], rotation=45)
ax.set(xlabel=f"{drv} decile", ylabel="mean AUROC gain from 180d", title=lab)
fig.tight_layout(); plt.show()
fig.tight_layout()
plt.show()
print(g.round(4).to_string())

## Q4 - How many patients are shared across cohorts?

In [ ]:
CACHE = Path(PROJECT_ROOT) / "saved_data" / "subject_sets.pkl"
cohorts_all = cc14.cohort_name.tolist()

if CACHE.exists():
    subs = pickle.load(open(CACHE, "rb"))
else:
    subs = {}
    for i, c in enumerate(cohorts_all):
        try:
            subs[c] = set(load_cohort(c).subject_id)
        except Exception:
            pass
        if i % 100 == 0:
            print(i, end=" ")
    pickle.dump(subs, open(CACHE, "wb"))

from collections import Counter
sizes = pd.Series({c: len(s) for c, s in subs.items()})
memb = Counter()
for s in subs.values():
    memb.update(s)
memb = pd.Series(memb)

print(f"cohorts {len(subs)} | sum of sizes {sizes.sum():,} | unique patients {len(memb):,} "
      f"| redundancy {sizes.sum()/len(memb):.2f}x")
print("patient membership counts:\n", memb.describe().round(2))
print("patients in >1 cohort:", f"{(memb>1).mean():.1%}", "| in >10:", f"{(memb>10).mean():.1%}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(memb, bins=np.arange(1, memb.max() + 2) - .5, log=True)
axes[0].set(xlabel="# cohorts a patient appears in", ylabel="patients (log)")

# pairwise Jaccard on a sample of cohort pairs
rng = np.random.default_rng(0)
names = list(subs)
pairs = [(names[i], names[j]) for i, j in rng.integers(0, len(names), (20000, 2)) if i != j]
jac = np.array([len(subs[x] & subs[y]) / len(subs[x] | subs[y]) for x, y in pairs])
axes[1].hist(jac, bins=60, log=True)
axes[1].set(xlabel="pairwise Jaccard", ylabel="pairs (log)")
print(f"\nJaccard: median {np.median(jac):.4f}  mean {jac.mean():.4f}  >0.1 in {(jac>0.1).mean():.1%} of pairs")
fig.tight_layout()
plt.show()

## Q5 - Within a D1 group, are the patients really similar?

In [ ]:
pair_df = pd.DataFrame(pairs, columns=["a", "b"]).assign(jac=jac)
pair_df["same_D1"] = pair_df.a.str.split("-").str[0] == pair_df.b.str.split("-").str[0]
pair_df["contain"] = [len(subs[x] & subs[y]) / min(len(subs[x]), len(subs[y])) for x, y in pairs]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].boxplot([pair_df[~pair_df.same_D1].jac, pair_df[pair_df.same_D1].jac],
                labels=["different D1", "same D1"], showfliers=False)
axes[0].set(ylabel="Jaccard", title="patient overlap")
axes[1].boxplot([pair_df[~pair_df.same_D1].contain, pair_df[pair_df.same_D1].contain],
                labels=["different D1", "same D1"], showfliers=False)
axes[1].set(ylabel="containment |A n B| / min(|A|,|B|)", title="nestedness")

demo = ["age", "female_rate", "death_rate", "n", "target_rate", "n_events_mean",
        "itembin_missing_mean", "cohens_d_adj_z"]
iccs = pd.Series({c: icc(cc14, c) for c in demo}).sort_values()
axes[2].barh(iccs.index, iccs.values)
axes[2].set(xlabel="variance between D1 groups (ICC)")
fig.tight_layout()
plt.show()

print(f"median Jaccard same D1 {pair_df[pair_df.same_D1].jac.median():.3f} vs "
      f"different D1 {pair_df[~pair_df.same_D1].jac.median():.4f}")
print(f"median containment same D1 {pair_df[pair_df.same_D1].contain.median():.3f}")
print(iccs.round(3))

In [ ]:
dup = pair_df[pair_df.jac > 0.6].sort_values("jac", ascending=False)
print(len(dup), "near-duplicate pairs;", dup.same_D1.mean().round(2), "share of them within the same D1")
print(dup.head(15).to_string())

In [ ]:
# needs `subs` from Q4
sel = sorted([c for c in subs if c.split("-")[0] in
              cc14.D1.value_counts().head(12).index])          # 12 largest D1 groups
M = np.zeros((len(sel), len(sel)))
for i, a in enumerate(sel):
    for j, b in enumerate(sel):
        M[i, j] = len(subs[a] & subs[b]) / len(subs[a] | subs[b])
np.fill_diagonal(M, np.nan)

d1s = [c.split("-")[0] for c in sel]
bounds = [k for k in range(1, len(sel)) if d1s[k] != d1s[k-1]]

fig, ax = plt.subplots(figsize=(11, 10))
im = ax.imshow(M, cmap="magma_r", vmin=0, vmax=np.nanpercentile(M, 99))
for b in bounds:
    ax.axhline(b - .5, color="c", lw=.6); ax.axvline(b - .5, color="c", lw=.6)
ax.set_xticks(range(len(sel)), sel, rotation=90, fontsize=5)
ax.set_yticks(range(len(sel)), sel, fontsize=5)
ax.set_title("patient overlap (Jaccard), cohorts grouped by D1")
fig.colorbar(im, shrink=.6); fig.tight_layout(); plt.show()

within = np.nanmean([M[i, j] for i in range(len(sel)) for j in range(len(sel)) if d1s[i] == d1s[j] and i != j])
across = np.nanmean([M[i, j] for i in range(len(sel)) for j in range(len(sel)) if d1s[i] != d1s[j]])
print(f"mean Jaccard within D1 {within:.3f} | across D1 {across:.4f} | ratio {within/max(across,1e-9):.1f}x")




In [ ]:
pool = {}
for c, s in subs.items():
    pool.setdefault(c.split("-")[0], set()).update(s)
top = sorted(pool, key=lambda k: -len(pool[k]))[:25]

J = np.array([[len(pool[a] & pool[b]) / len(pool[a] | pool[b]) for b in top] for a in top])
np.fill_diagonal(J, np.nan)

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(J, cmap="magma_r")
ax.set_xticks(range(len(top)), top, rotation=90); ax.set_yticks(range(len(top)), top)
ax.set_title("patient overlap between D1 groups")
fig.colorbar(im, shrink=.6); fig.tight_layout(); plt.show()
print(f"D1-to-D1 Jaccard: median {np.nanmedian(J):.3f}, max {np.nanmax(J):.3f}")


## Q6 - Grouped by D1 (same patients, same features): is the signal driven by the label?

In [ ]:
run = "14d agg24 VMD"
d6 = (cc14.merge(res[res.run == run][["cohort_name", "auroc", "auprc"]], on="cohort_name")
          .dropna(subset=["auroc"]))
d6["auprc_norm"] = d6.auprc / d6.target_rate

for col in ["auroc", "auprc_norm", "cohens_d_adj_z", "target_rate"]:
    print(f"ICC by D1  {col:16s} {icc(d6, col):.3f}")

big = d6.groupby("D1").filter(lambda g: len(g) >= 5)
order = big.groupby("D1").auroc.median().sort_values().index

fig, axes = plt.subplots(2, 1, figsize=(13, 8))
axes[0].boxplot([big[big.D1 == k].auroc for k in order], labels=list(order), showfliers=False)
axes[0].axhline(0.5, color="k", ls="--", lw=1)
axes[0].tick_params(axis="x", rotation=90)
axes[0].set(ylabel="AUROC", title=f"AUROC per D1 group ({run})")

rows = []
for k, g in big.groupby("D1"):
    rows.append({"D1": k, "n_cohorts": len(g),
                 "auroc_spread": g.auroc.max() - g.auroc.min(),
                 "rho_cohens": spearmanr(g.cohens_d_adj_z, g.auroc, nan_policy="omit").statistic,
                 "rho_target": spearmanr(g.target_rate, g.auroc, nan_policy="omit").statistic})
wi = pd.DataFrame(rows)

axes[1].hist(wi.rho_cohens.dropna(), bins=20)
axes[1].axvline(0, color="k", ls="--", lw=1)
axes[1].axvline(wi.rho_cohens.median(), color="r", lw=1)
axes[1].set(xlabel="within-D1 spearman(cohens_d_adj_z, AUROC)", ylabel="D1 groups")
fig.tight_layout()
plt.show()

print(f"\nwithin-D1 AUROC spread: median {wi.auroc_spread.median():.3f} (max {wi.auroc_spread.max():.3f})")
print(f"median within-D1 rho with cohens_d_adj_z {wi.rho_cohens.median():+.3f} "
      f"({(wi.rho_cohens>0).mean():.0%} of groups positive)")
print(f"median within-D1 rho with target_rate    {wi.rho_target.median():+.3f}")

In [ ]:
w = wi.dropna(subset=["rho_cohens"]).sort_values("rho_cohens")
fig, ax = plt.subplots(figsize=(6, 0.18 * len(w)))
ax.barh(w.D1, w.rho_cohens)
ax.axvline(0, color="k", lw=1)
ax.set(xlabel="within-D1 spearman(cohens_d_adj_z, AUROC)")
fig.tight_layout(); plt.show()


In [ ]:
sizes_d1 = d6.D1.value_counts()
print(f"{d6.D1.nunique()} D1 groups total | {(sizes_d1 >= 5).sum()} with >=5 cohorts "
      f"| covering {sizes_d1[sizes_d1 >= 5].sum()} of {len(d6)} cohorts")
print(sizes_d1.describe())


## Q7 - Performance vs each characteristic (collinearity checked, D1-clustered CIs)

In [ ]:
CHARS = [c for c in cc14.select_dtypes("number").columns
         if c not in ("seed", "fold", "agg_int", "days_before_discharge")]

cm = cc14[CHARS].corr("spearman")
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(cm, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(CHARS)), CHARS, rotation=90, fontsize=7)
ax.set_yticks(range(len(CHARS)), CHARS, fontsize=7)
fig.colorbar(im)
ax.set_title("characteristic collinearity (spearman)")
fig.tight_layout()
plt.show()

print("target_rate vs n:      ", round(cm.loc["target_rate", "n"], 3))
print("n vs n_positive:       ", round(cm.loc["n", "n_positive"], 3))
print("n vs counts_cohort:    ", round(cm.loc["n", "counts_cohort"], 3))

In [ ]:
MET = ["auroc", "auprc", "auprc_norm", "f1", "mcc"]

def char_table(run, cc, B=300):
    d = cc.merge(res[res.run == run][["cohort_name"] + METRICS], on="cohort_name")

    d["auprc_norm"] = d.auprc / d.target_rate
    out = []
    for f in CHARS:
        if d[f].nunique() < 3:
            continue
        for m in MET:
            ci = cboot(d, lambda s, f=f, m=m: spearmanr(s[f], s[m], nan_policy="omit").statistic, B=B)
            ctrl = [c for c in ("n", "target_rate") if c != f]
            out.append({"feature": f, "metric": m, "rho": ci[1], "lo": ci[0], "hi": ci[2],
                        "robust": ci[0] * ci[2] > 0, "rho_partial": pspear(d, f, m, ctrl)})
    return pd.DataFrame(out)

tab14 = char_table("14d agg24 VMD", cc14)

piv  = tab14.pivot(index="feature", columns="metric", values="rho").reindex(columns=MET)
mask = tab14.pivot(index="feature", columns="metric", values="robust").reindex(columns=MET)
part = tab14.pivot(index="feature", columns="metric", values="rho_partial").reindex(columns=MET)
piv  = piv.reindex(piv.abs().max(axis=1).sort_values(ascending=False).index)

fig, axes = plt.subplots(1, 2, figsize=(12, 10), sharey=True)
for ax, M, t in [(axes[0], piv, "spearman rho (* = D1-clustered CI excludes 0)"),
                 (axes[1], part.loc[piv.index], "partial (n, target_rate removed)")]:
    im = ax.imshow(M, cmap="coolwarm", vmin=-.6, vmax=.6)
    ax.set_xticks(range(len(MET)), MET, rotation=45)
    ax.set_title(t, fontsize=9)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M.iloc[i, j]
            if pd.notna(v):
                star = "*" if (t.startswith("spearman") and mask.loc[M.index[i], MET[j]]) else ""
                ax.text(j, i, f"{v:.2f}{star}", ha="center", va="center", fontsize=6)
axes[0].set_yticks(range(len(piv)), piv.index, fontsize=7)
fig.colorbar(im, ax=axes, shrink=.5)
plt.show()

robust_auroc = tab14[(tab14.metric == "auroc") & tab14.robust].copy()
print(robust_auroc.reindex(robust_auroc.rho.abs().sort_values(ascending=False).index).round(3).to_string())

In [ ]:
# same table for the count and stats runs
tab_count = char_table("180d 1440h count", cc180)
tab_stats = char_table("180d 1440h stats", cc180)

for name, t in [("180d count", tab_count), ("180d stats", tab_stats)]:
    r = t[(t.metric == "auroc") & t.robust]
    print(f"\n=== {name}: characteristics robustly linked to AUROC ===")
    print(r.reindex(r.rho.abs().sort_values(ascending=False).index).round(3).to_string())

In [ ]:
COHORTSSINA = ["F10-K63","M25-D47", "M48-M96"]

In [ ]:
res = pd.concat([v.assign(run=k) for k, v in runs.items()], ignore_index=True)
res = res.drop_duplicates(["cohort_name", "run"], keep="last")
res["D1"] = res.cohort_name.str.split("-").str[0]
AUROC = res.pivot(index="cohort_name", columns="run", values="auroc")

sub = res[res.cohort_name.isin(COHORTSSINA)]
COHORTS_PLOT = [c for c in COHORTSSINA if c in set(sub.cohort_name)]   # drops F10-K63


In [ ]:
COHORTSSINA = ["M25-D47", "M48-M96"]
PLOT_METRICS = ["auroc", "auprc", "f1", "mcc", "recall", "precision"]

sub = res[res.cohort_name.isin(COHORTSSINA)]

fig, axes = plt.subplots(1, len(COHORTSSINA), figsize=(5 * len(COHORTSSINA), 4.5), sharey=True)
x = np.arange(len(PLOT_METRICS))
w = 0.8 / len(runs)

for ax, c in zip(axes, COHORTSSINA):
    d = sub[sub.cohort_name == c].set_index("run").reindex(list(runs))
    for i, (r, color) in enumerate(zip(runs, plt.cm.tab10.colors)):
        ax.bar(x + i * w - 0.4 + w / 2, d.loc[r, PLOT_METRICS].values, w,
               label=r, color=color)
    ax.axhline(0.5, color="k", ls="--", lw=.8)
    ax.set_xticks(x, PLOT_METRICS, rotation=45, ha="right")
    ax.set_title(c)
    ax.grid(axis="y", lw=.6, alpha=.3)
    ax.set_axisbelow(True)

axes[0].set_ylabel("score")
axes[-1].legend(frameon=False, fontsize=8)
fig.tight_layout()
plt.show()

print(sub.pivot(index="run", columns="cohort_name", values="auroc").reindex(list(runs)).round(3).to_string())


In [ ]:
print([c for c in COHORTSSINA if c in set(rep_cohorts.cohort_name)])   # likely []
print(res[res.cohort_name.isin(COHORTSSINA)].shape)                    # (0, ...)


In [ ]:
coho

## ML models new Runs

In [ ]:
COHORTS = selected_cohorts.cohort_name
COHORTS.tolist()
#

COHORTSSINA = ["F10-K63","M25-D47", "M48-M96"]

In [ ]:
selected_cohorts = rep_cohorts.iloc[[10,15,67,55,99]]
selected_cohorts

In [ ]:
import argparse
from pathlib import Path
from flab_training.envmanager import EnvManager
from config.constants import PROJECT_ROOT

#selected_cohorts = rep_cohorts.iloc[[10,15,67,55,99]]

COHORTS = ['C25-K59'] #COHORTSSINA #['C64-C78', 'K44-K59', 'I12-K65', 'E78-G81']#selected_cohorts.cohort_name

for cohort in COHORTS:
    print(f"\n=== {cohort} ===")
    args = argparse.Namespace(
        days_before_discharge=14,
        dataset="MIMIC_IV",
        cohort=cohort,
        fold=0,
        model_type="random_forest",
        grid="nested",
        pretrain=False,
        feature_selection=True,
        feature_selection_method="mimic_top_100",
        load_ckpt_path=None,
        prefix="test_no_ov",
        static_threshold=0,
        hid_dim_demo=64,
        agg_int=24, # hours 
        drop_minutes=False,
        freeze=False,
        config_path=str(Path(PROJECT_ROOT) / "flab_training" / "config_params.yaml"),
        split_seed=None,
        fast=False,
        search="grid",
        n_trials=25,
        impute="fill",
        variant="VMD",
        extractor="DTB",
        first_adm_only=True,
        oversampling=None,
        agg = "mean",
        feature_combination_method = "concatenate",
    )
    EnvManager(args).train_full()


In [ ]:
# 07.09 runs
hpc_run_dir = Path(PROJECT_ROOT) / "saved_data"  / "results_0709"
prefix = "test"
agg = 24
variant = "vmd"
hpc_run_results = load_results(COHORTS, prefix, agg, "VMD", results_dir=hpc_run_dir)
hpc_run_results

In [ ]:
# 07.09 runs
hpc_run_dir = Path(PROJECT_ROOT) / "saved_data"  / "results_0709"
prefix = "test_no_ov"
agg = 24
variant = "vmd"
hpc_run_results = load_results(COHORTS, prefix, agg, "VMD", results_dir=hpc_run_dir)
hpc_run_results

In [ ]:
hpc_run_dir = Path(PROJECT_ROOT) / "saved_data"  / "results"
prefix = "first_adms_14d_mimic100"
agg = 24
variant = "VMD"
results_14d =  load_results(rep_cohorts.cohort_name.tolist(), prefix, agg, variant, results_dir=hpc_run_dir)


In [ ]:
hpc_run_dir = Path(PROJECT_ROOT) / "saved_data"  / "results"
prefix = "stats_180d"
agg = 1440
variant = "stats"

results_stats_180d = load_results(rep_cohorts.cohort_name.tolist(), prefix, agg, variant, results_dir=hpc_run_dir)


In [ ]:
from scipy.stats import wilcoxon

a = results_14d.set_index("cohort_name")[METRICS].astype(float)
b = results_stats_180d.set_index("cohort_name")[METRICS].astype(float)
common = a.index.intersection(b.index)
a, b = a.loc[common], b.loc[common]
d = b - a                                 
pd.DataFrame({
    "14d_VMD":    a.mean(),
    "180d_stats": b.mean(),
    "mean_delta": d.mean(),
    "stats_wins": (d > 0).sum(),
    "p":          [wilcoxon(b[m], a[m], zero_method="zsplit").pvalue for m in METRICS],
}).round(4)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

PLOT_METRICS = ["auroc", "auprc", "f1"]
LABELS  = {"auroc": "AUROC", "auprc": "AUPRC", "f1": "F1"}
NAMES   = ["14d VMD", "180d stats"]
COLORS  = ["#3b6ea5", "#d1802f"]          # blue / orange: safe for CVD
INK, MUTED, GRID = "#1a1a1a", "#5c5c5c", "#d8d8d8"

a = results_14d.set_index("cohort_name")
b = results_stats_180d.set_index("cohort_name")
common = a.index.intersection(b.index)
a, b = a.loc[common, PLOT_METRICS].astype(float), b.loc[common, PLOT_METRICS].astype(float)
print(f"{len(common)} paired cohorts")


def style(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color(GRID)
    ax.yaxis.grid(True, color=GRID, lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(colors=MUTED, length=0)


def box(ax, data, positions, colors):
    bp = ax.boxplot(data, positions=positions, widths=0.5, patch_artist=True,
                    showfliers=False, zorder=3,
                    medianprops=dict(color=INK, lw=2),
                    whiskerprops=dict(color=MUTED, lw=1),
                    capprops=dict(color=MUTED, lw=1),
                    boxprops=dict(lw=0))
    for patch, c in zip(bp["boxes"], colors):
        patch.set_facecolor(c)
        patch.set_alpha(0.8)


# ---- Figure 1: the two configs side by side, with paired lines ----
fig, axes = plt.subplots(1, 3, figsize=(10.5, 4.2))
for ax, m in zip(axes, PLOT_METRICS):
    x, y = a[m].values, b[m].values
    p = wilcoxon(y, x, zero_method="zsplit").pvalue

    for xi, yi in zip(x, y):                       # one faint line per cohort
        ax.plot([1, 2], [xi, yi], color="0.8", lw=0.4, alpha=0.5, zorder=1)
    box(ax, [x, y], [1, 2], COLORS)

    ax.set_xticks([1, 2], NAMES, color=INK)
    ax.set_title(LABELS[m], color=INK, fontsize=12, pad=10)
    sig = "n.s." if p >= 0.05 else f"p = {p:.3g}"
    ax.text(0.5, 1.005, sig, transform=ax.transAxes, ha="center", va="bottom",
            color=MUTED, fontsize=9)
    style(ax)

axes[0].set_ylabel("score", color=MUTED)
fig.suptitle(f"14d VMD vs 180d stats — random forest, fold 0, {len(common)} cohorts",
             color=INK, fontsize=13, y=1.02)
fig.tight_layout()
plt.show()


# ---- Figure 2: per-cohort difference (what the test measures) ----
d = b - a
fig, ax = plt.subplots(figsize=(6.2, 4.2))
ax.axhline(0, color=MUTED, lw=1.2, zorder=2)
box(ax, [d[m].values for m in PLOT_METRICS], [1, 2, 3], ["#8a8a8a"] * 3)
for i, m in enumerate(PLOT_METRICS, start=1):      # jittered points
    xj = np.random.default_rng(0).normal(i, 0.055, len(d))
    ax.scatter(xj, d[m], s=9, color=COLORS[1], alpha=0.45, lw=0, zorder=4)

ax.set_xticks([1, 2, 3], [LABELS[m] for m in PLOT_METRICS], color=INK)
ax.set_ylabel("180d stats − 14d VMD", color=MUTED)
ax.set_title("Per-cohort difference (above 0 = 180d stats better)",
             color=INK, fontsize=12, pad=10)
style(ax)
fig.tight_layout()
plt.show()


# ---- The tests ----
rows = []
for m in PLOT_METRICS:
    res = wilcoxon(b[m], a[m], zero_method="zsplit")
    rows.append({
        "metric": LABELS[m],
        "14d_VMD": a[m].mean(),
        "180d_stats": b[m].mean(),
        "mean_delta": d[m].mean(),
        "median_delta": d[m].median(),
        "stats_wins": int((d[m] > 0).sum()),
        "ties": int((d[m] == 0).sum()),
        "p_wilcoxon": res.pvalue,
        "p_bonferroni": min(res.pvalue * len(PLOT_METRICS), 1.0),
    })
pd.DataFrame(rows).set_index("metric").round(4)


In [ ]:
main_dir = Path(PROJECT_ROOT) / "saved_data" / "results"

In [ ]:
COHORTS = selected_cohorts.cohort_name
COHORTS = COHORTSSINA

In [ ]:
main_dir = Path(PROJECT_ROOT) / "saved_data" / "results"
prefix = "test_count"
agg = 1440
variant = "count"
for model in ["random_forest","catboost"]:
    new_results_stats = load_results(COHORTS, prefix, agg, variant, models=[model], results_dir=main_dir)
    print(new_results_stats)


In [ ]:
main_dir = Path(PROJECT_ROOT) / "saved_data" / "results"
prefixes = ["test","test_3"]
agg = 1440
variant = "stats"
for model in ["random_forest","lstm","gru"]:#,"catboost","xgboost"]:
    for prefix in prefixes:
        new_results_stats = load_results(COHORTS, prefix, agg, variant, models=[model], results_dir=main_dir)
        print(model,prefix)
        print(new_results_stats)


In [ ]:
main_dir = Path(PROJECT_ROOT) / "saved_data" / "results"
prefix = "test_14_daily"
agg = 24
variant = "VMD"
new_results_stats = load_results(COHORTS, prefix, agg, variant, models=["random_forest"], results_dir=main_dir)
new_results_stats

In [ ]:
main_dir = Path(PROJECT_ROOT) / "saved_data" / "results"
prefix = "first_adms_14d_mrmr100"
agg = 24
variant = "VMD"
new_results_stats = load_results(COHORTS, prefix, agg, variant, models=["random_forest"], results_dir=main_dir)
new_results_stats

In [ ]:
main_dir = Path(PROJECT_ROOT) / "saved_data" / "results"
prefix = "initial_run"
agg = 24
variant = "VMD"
new_results_stats = load_results(COHORTS, prefix, agg, variant, models=["random_forest"], results_dir=main_dir)
new_results_stats


In [ ]:
main_dir = Path(PROJECT_ROOT) / "saved_data" / "results"
prefix = "initial_run_180_d"
agg = 24
variant = "VMD"
new_results_stats = load_results(COHORTS, prefix, agg, variant, models=["random_forest"], results_dir=main_dir)
new_results_stats

In [ ]:
main_dir = Path(PROJECT_ROOT) / "saved_data" / "results"
agg = 1440
variant = "stats"
#prefixes  = ["stats_180_3bins","stats_180_3bins_cross_features","stats_180_3bins_new_processor","stats_180_3bins_new_processor_old_trim"]
prefixes  = ["stats_180_3bins_new_processor_old_trim","stats_180_3bins_new_processor_ohne_trim"]
for p in prefixes: 
    new_results_stats = load_results(COHORTS, p, agg, variant, models=["random_forest"], results_dir=main_dir)
    print(p ,new_results_stats)

# Plot

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

main_dir = Path(PROJECT_ROOT) / "saved_data" / "results"
METRICS = ["auroc", "auprc", "f1", "mcc", "recall", "precision"]

CONFIGS = {
    "180d_3bin_stats":         dict(prefix="stats_180_3bins_new_processor_old_trim",  agg=1440, variant="stats", days=180),
    "180d_3bin_stats_no_trim": dict(prefix="stats_180_3bins_new_processor_ohne_trim", agg=1440, variant="stats", days=180),
    "initial_14d_vmd":         dict(prefix="initial_run",                             agg=24,   variant="VMD",   days=14),
    "initial_180d_vmd":        dict(prefix="initial_run_180_d",                       agg=24,   variant="VMD",   days=180),
}
MODEL, FOLD = "random_forest", 0

COLORS = plt.cm.tab10(np.linspace(0, 1, 10))

def leaf_path(cohort, cfg, results_dir):
    return (results_dir / cohort / MODEL / cfg["prefix"] / f"fold_{FOLD}" /
            f"agg_int_{cfg['agg']}" / "impute_fill" / f"variant_{cfg['variant']}" / "results_final.csv")

def load_configs(cohorts, configs, results_dir):
    rows, missing = [], []
    for cohort in cohorts:
        for label, cfg in configs.items():
            f = leaf_path(cohort, cfg, results_dir)
            if not f.exists():
                missing.append(f)
                continue
            rows.append(pd.read_csv(f).assign(cohort_name=cohort, config=label, **cfg))
    if missing:
        print(f"{len(missing)} missing, e.g. {missing[0]}")
    if not rows:
        return pd.DataFrame()
    df = pd.concat(rows, ignore_index=True)
    df[METRICS] = df[METRICS].apply(pd.to_numeric, errors="coerce")
    return df[["cohort_name", "config", "prefix", "agg", "variant", "days"] + METRICS]

all_cohorts = COHORTSSINA #sorted(d.name for d in main_dir.iterdir() if d.is_dir() and d.name != "cohort_name")
COHORTS = [c for c in all_cohorts if all(leaf_path(c, cfg, main_dir).exists() for cfg in CONFIGS.values())]
print(f"{len(COHORTS)} cohorts with all {len(CONFIGS)} configs: {COHORTS}")

combined = load_configs(COHORTS, CONFIGS, main_dir)

wide = combined.pivot(index="cohort_name", columns="config", values="auroc")
wide = wide[list(CONFIGS)].dropna().sort_values(list(CONFIGS)[0])

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(wide))
for v, c in zip(CONFIGS, COLORS):
    ax.plot(x, wide[v], marker="o", ms=4, lw=1, color=c, label=v, alpha=0.85)

ax.set_xticks(x, wide.index, rotation=90, fontsize=6)
ax.axhline(0.5, color="#52514e", lw=1, ls=(0, (4, 3)))
ax.set_ylabel("AUROC")
ax.set_title("Per-cohort AUROC across configs")
ax.legend(frameon=False)
ax.grid(axis="y", lw=0.7, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path(PROJECT_ROOT) / "saved_data"
METRICS = ["auroc", "auprc", "f1", "mcc", "recall", "precision"]

'''COHORTS = ["C25-K59", "C64-C78", "K44-K59", "I12-K65", "E78-G81"]
MODEL, FOLD = "random_forest", 0

CONFIGS = {
    "VMD 180d":       dict(root="results_trials",      prefix="test_3108_stats_180_3bin_ov", agg=24,   variant="VMD"),
    "STATS 180d":     dict(root="results_trials",      prefix="stats_180d_3bin_ov",          agg=1440, variant="STATS"),
    "stats new proc": dict(root="results",             prefix="stats_180_3bins_new_processor_old_trim", agg=1440, variant="stats"),
    "VMD 14d":        dict(root="hpc_run_3008/results", prefix="first_adms_14d_mimic100",    agg=24,   variant="VMD"),
}'''

COHORTS = ["C91-D63", "E05-E89", "J45-J38", "I73-J81", "O26-O34"]

CONFIGS = {
    "180d mimic100":  dict(root="results_trials",       prefix="test_3108_stats_180_3bin_ov",          agg=24, variant="VMD"),
    "180d newfeat":   dict(root="results_trials",       prefix="test_3108_stats_180_3bins_ov_newfeat", agg=24, variant="VMD"),
    "360d 4bin":      dict(root="results_trials",       prefix="test_3108_stats_360_4bin_ov",          agg=24, variant="VMD"),
    "14d mimic100":   dict(root="hpc_run_3008/results", prefix="first_adms_14d_mimic100",              agg=24, variant="VMD"),
    "14d mrmr100":    dict(root="hpc_run_3008/results", prefix="first_adms_14d_mrmr100",               agg=24, variant="VMD"),
}


def leaf_path(cohort, cfg):
    return (ROOT / cfg["root"] / cohort / MODEL / cfg["prefix"] / f"fold_{FOLD}" /
            f"agg_int_{cfg['agg']}" / "impute_fill" / f"variant_{cfg['variant']}" / "results_final.csv")

rows, missing = [], []
for cohort in COHORTS:
    for label, cfg in CONFIGS.items():
        f = leaf_path(cohort, cfg)
        if not f.exists():
            missing.append(f)
            continue
        rows.append(pd.read_csv(f).assign(cohort_name=cohort, config=label))
if missing:
    print(f"{len(missing)} missing, e.g. {missing[0]}")

combined = pd.concat(rows, ignore_index=True)
combined[METRICS] = combined[METRICS].apply(pd.to_numeric, errors="coerce")

wide = combined.pivot(index="cohort_name", columns="config", values="auroc")
wide = wide[list(CONFIGS)].dropna().sort_values(list(CONFIGS)[0])

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(wide))
for v, c in zip(CONFIGS, plt.cm.tab10.colors):
    ax.plot(x, wide[v], marker="o", ms=6, lw=1.2, color=c, label=v, alpha=0.85)

ax.set_xticks(x, wide.index, rotation=45, ha="right")
ax.axhline(0.5, color="#52514e", lw=1, ls=(0, (4, 3)))
ax.set_ylabel("AUROC")
ax.set_title("Per-cohort AUROC across variants")
ax.legend(frameon=False)
ax.grid(axis="y", lw=0.7, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path(PROJECT_ROOT) / "saved_data"
METRICS = ["auroc", "auprc", "f1", "mcc", "recall", "precision"]
MODEL, FOLD = "random_forest", 0

COHORTS = ["C25-K59", "C64-C78", "K44-K59", "I12-K65", "E78-G81",
           "C91-D63", "E05-E89", "J45-J38", "I73-J81", "O26-O34"]

CONFIGS = {
    "180d stats": dict(root="results_trials",       prefix="test_3108_stats_180_3bin_ov",          agg=24, variant="VMD"),
    #"180d newfeat":  dict(root="results_trials",       prefix="test_3108_stats_180_3bins_ov_newfeat", agg=24, variant="VMD"),
    "14d mimic100":  dict(root="hpc_run_3008/results", prefix="first_adms_14d_mimic100",              agg=24, variant="VMD"),
    "14d mrmr100":   dict(root="hpc_run_3008/results", prefix="first_adms_14d_mrmr100",               agg=24, variant="VMD"),
}

def leaf_path(cohort, cfg):
    return (ROOT / cfg["root"] / cohort / MODEL / cfg["prefix"] / f"fold_{FOLD}" /
            f"agg_int_{cfg['agg']}" / "impute_fill" / f"variant_{cfg['variant']}" / "results_final.csv")

rows, missing = [], []
for cohort in COHORTS:
    for label, cfg in CONFIGS.items():
        f = leaf_path(cohort, cfg)
        if not f.exists():
            missing.append(f)
            continue
        rows.append(pd.read_csv(f).assign(cohort_name=cohort, config=label))
if missing:
    print(f"{len(missing)} missing, e.g. {missing[0]}")

combined = pd.concat(rows, ignore_index=True)
combined[METRICS] = combined[METRICS].apply(pd.to_numeric, errors="coerce")

for metric in ["auroc"]:
    wide = combined.pivot(index="cohort_name", columns="config", values=metric)
    wide = wide.reindex(index=COHORTS, columns=list(CONFIGS))
    wide.loc["MEAN"] = wide.mean()
    print(f"\n=== {metric.upper()} ===")
    print(wide.round(3).to_string())


In [ ]:
wide

In [ ]:
cohort_cc[cohort_cc.cohort_name.isin(COHORTS)].merge(wide,on="cohort_name")